Dataset derivation

1. Uses Nemotron Personas for personalities
2. Uses 2021 M&P Survey as the baseline for the splits


Step 1: Import dataset from Nemotron

In [1]:
import pandas as pd
import numpy as np

from datasets import load_dataset

ds = load_dataset("nvidia/Nemotron-Personas-Singapore")

print(ds)
print(ds.keys())
print(ds.num_rows)
print(ds.column_names)


c:\Users\chong\OneDrive - Singapore Management University\research\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['uuid', 'professional_persona', 'sports_persona', 'arts_persona', 'travel_persona', 'culinary_persona', 'persona', 'cultural_background', 'skills_and_expertise', 'skills_and_expertise_list', 'hobbies_and_interests', 'hobbies_and_interests_list', 'career_goals_and_ambitions', 'sex', 'age', 'marital_status', 'education_level', 'occupation', 'industry', 'planning_area', 'country'],
        num_rows: 148000
    })
})
dict_keys(['train'])
{'train': 148000}
{'train': ['uuid', 'professional_persona', 'sports_persona', 'arts_persona', 'travel_persona', 'culinary_persona', 'persona', 'cultural_background', 'skills_and_expertise', 'skills_and_expertise_list', 'hobbies_and_interests', 'hobbies_and_interests_list', 'career_goals_and_ambitions', 'sex', 'age', 'marital_status', 'education_level', 'occupation', 'industry', 'planning_area', 'country']}


In [2]:
data = ds["train"]
df = data.to_pandas()

df.head()

,uuid,professional_persona,sports_persona,arts_persona,travel_persona,culinary_persona,persona,cultural_background,skills_and_expertise,skills_and_expertise_list,...,hobbies_and_interests_list,career_goals_and_ambitions,sex,age,marital_status,education_level,occupation,industry,planning_area,country
0,d792702ec018494e8e49c69120759408,"Yi Peng Yong, known as Danelle, a 20‑year‑old ...","Yi Peng Yong, known as Danelle, enjoys playing...","Yi Peng Yong, known as Danelle, spends her eve...","Yi Peng Yong, known as Danelle, meticulously p...","Yi Peng Yong, known as Danelle, delights in co...","Yi Peng Yong, known as Danelle, blends a love ...",Yi Peng Yong grew up in the multicultural neig...,She has developed solid expertise in wholesale...,"['Inventory management', 'Supply chain coordin...",...,['Reading personal development and business bo...,She aims to deepen her knowledge of retail ope...,Female,20,Single,Polytechnic,Senior Official or Manager,Wholesale & Retail Trade,Sengkang,Singapore
1,2fe149f1c59e47daa5e6ae0881ffef93,"Charmaine, a 53‑year‑old experienced clerical ...","Charmaine, a devoted fan of Brunei DPMM FC, fo...","Charmaine, an avid admirer of Jack Neo’s comed...","Charmaine, who dreams of a serene Japan onsen ...","Charmaine, a passionate home cook, delights in...","Charmaine, a sociable 53‑year‑old office whiz ...",Charmaine grew up in a bilingual Chinese house...,Charmaine is an experienced clerical worker wi...,"['Data entry', 'Inventory management', 'Custom...",...,"['Karaoke', 'Cooking Cantonese cuisine', 'Gard...",Charmaine aims to advance to a senior administ...,Female,53,Married,Secondary,Clerical Worker,Wholesale & Retail Trade,Choa Chu Kang,Singapore
2,dd95514790e84ffa9973475cfc706660,"Betty, a 29‑year‑old policy analyst in the Min...",Betty is an avid follower of Red Bull Racing’s...,Betty’s artistic palate is shaped by her love ...,Betty meticulously plans her cherry‑blossom pi...,Betty excels at preparing classic Chinese dish...,Betty is a methodical policy pro who balances ...,Wei Qi Leong grew up in a middle‑class Chinese...,Betty has developed strong analytical and orga...,"['Policy analysis', 'Program management', 'Dat...",...,"['Reading non-fiction', 'Chinese calligraphy',...",Betty aims to advance to a senior policy advis...,Female,29,Single,University,Professional,Public Administration & Education Services,Clementi,Singapore
3,a2d66162e29f420aa93c670a41eb4fd2,"Jian Choy Ker, a retired hawker stall propriet...","Jian Choy Ker, though in his nineties, stays c...",Jian Choy Ker immerses himself in classic Cant...,Jian Choy Ker prefers relaxed staycations with...,"Jian Choy Ker, a seasoned hawker‑style cook, d...","Jian Choy Ker, a 91‑year‑old former hawker who...",Jian Choy Ker grew up in a traditional Chinese...,Through decades of work as a hawker stall prop...,"['Mandarin and Cantonese fluency', 'Basic Engl...",...,"['Mahjong', 'Chinese calligraphy', 'Gardening'...","Although retired, Jian aims to maintain good h...",Male,91,Married,Primary,Retired,NaN,Sengkang,Singapore
4,98a3e5752be446ebaca994b8a390b290,"Aixin Noy Yong, a dedicated homemaker turned c...","Aixin Noy Yong, a quiet admirer of the Golden ...","Aixin Noy Yong, an ardent fan of Andy Lau, spe...",Aixin Noy Yong dreams of a serene Japanese ons...,"Aixin Noy Yong, a master of traditional cookin...","Aixin Noy Yong, a 68‑year‑old tea‑tin collecto...",Aixin grew up in a traditional Chinese Singapo...,"Aixin is adept at traditional Chinese cooking,...","['Chinese culinary arts (dim sum, herbal soups...",...,"['Reading Buddhist texts', 'Listening to class...","Although she identifies as a homemaker, Aixin ...",Female,68,Single,Other Diploma,Homemaker,NaN,Kallang,Singapore


Step 2:
Extract our direct fields from dataset that agents need.

In [3]:
cols = [
    "age",
    "sex",
    "marital_status",
    "education_level",
    "occupation",
    "industry",
    "planning_area",
    "persona",
    "cultural_background",
    "career_goals_and_ambitions"
]

# Check that all required columns exist before selecting them
missing_cols = [c for c in cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns in Nemotron dataset: {missing_cols}")

df = df[cols].copy()

# Keep original dataset row index for traceability
df["source_index"] = df.index

# Rename raw fields so cleaned/modelled fields are separate
df = df.rename(columns={
    "sex": "gender",
})

df.head()


,age,gender,marital_status,education_level,occupation,industry,planning_area,persona,cultural_background,career_goals_and_ambitions,source_index
0,20,Female,Single,Polytechnic,Senior Official or Manager,Wholesale & Retail Trade,Sengkang,"Yi Peng Yong, known as Danelle, blends a love ...",Yi Peng Yong grew up in the multicultural neig...,She aims to deepen her knowledge of retail ope...,0
1,53,Female,Married,Secondary,Clerical Worker,Wholesale & Retail Trade,Choa Chu Kang,"Charmaine, a sociable 53‑year‑old office whiz ...",Charmaine grew up in a bilingual Chinese house...,Charmaine aims to advance to a senior administ...,1
2,29,Female,Single,University,Professional,Public Administration & Education Services,Clementi,Betty is a methodical policy pro who balances ...,Wei Qi Leong grew up in a middle‑class Chinese...,Betty aims to advance to a senior policy advis...,2
3,91,Male,Married,Primary,Retired,NaN,Sengkang,"Jian Choy Ker, a 91‑year‑old former hawker who...",Jian Choy Ker grew up in a traditional Chinese...,"Although retired, Jian aims to maintain good h...",3
4,68,Female,Single,Other Diploma,Homemaker,NaN,Kallang,"Aixin Noy Yong, a 68‑year‑old tea‑tin collecto...",Aixin grew up in a traditional Chinese Singapo...,"Although she identifies as a homemaker, Aixin ...",4


Step 3: Filter out age
Step 4: create age buckets for filtering

In [4]:
df = df[(df["age"] >= 21) & (df["age"] <= 45)].copy()
df.head()

def age_bucket(age):
    if 21 <= age <= 25:
        return "21-25"
    elif 26 <= age <= 30:
        return "26-30"
    elif 31 <= age <= 35:
        return "31-35"
    elif 36 <= age <= 40:
        return "36-40"
    elif 41 <= age <= 45:
        return "41-45"
    else:
        return np.nan

df["age_bucket"] = df["age"].apply(age_bucket)
df.head()

,age,gender,marital_status,education_level,occupation,industry,planning_area,persona,cultural_background,career_goals_and_ambitions,source_index,age_bucket
2,29,Female,Single,University,Professional,Public Administration & Education Services,Clementi,Betty is a methodical policy pro who balances ...,Wei Qi Leong grew up in a middle‑class Chinese...,Betty aims to advance to a senior policy advis...,2,26-30
5,39,Male,Married,University,Senior Official or Manager,Manufacturing,Sengkang,"Qijie Huat Chua, a night-owl puzzle enthusiast...",Qijie Huat Chua grew up in a Singaporean Chine...,He aims to advance to a C‑suite role such as C...,5,36-40
6,42,Female,Married,Post Secondary (Non-Tertiary),Homemaker,NaN,Jurong West,Wai Ting Soon is a habit‑driven planner who ca...,Wai Ting grew up in a traditional Chinese Sing...,While she values her role as a full‑time homem...,6,41-45
7,42,Male,Single,University,Professional,Financial & Insurance Services,Queenstown,Chongyong Ang balances a spreadsheet‑obsessed ...,Chongyong grew up in a typical Chinese Singapo...,Chongyong aims to progress to a senior managem...,7,41-45
9,31,Male,Single,University,Senior Official or Manager,Wholesale & Retail Trade,Woodlands,Yao Dar Teo (Daniel) is a data‑savvy retail st...,"Daniel grew up in Woodlands, a residential tow...",Daniel aims to advance to a senior director or...,9,31-35


Step 4: Standardise Education

In [5]:
print("Unique values in education_level_raw before cleaning:", df["education_level"].unique())

def clean_education(x):
    x = str(x).lower()

    if x in ["secondary", "primary", "lower secondary", "no qualification"]:
        return "Secondary and below"

    elif x in ["post secondary (non-tertiary)", "other diploma", "polytechnic"]:
        return "Diploma / A-Level"

    elif x in ["university"]:
        return "Degree and above"

df["education_level_processed"] = df["education_level"].apply(clean_education)
df.head()

Unique values in education_level_raw before cleaning: <ArrowStringArray>
[                   'University', 'Post Secondary (Non-Tertiary)',
                     'Secondary',                 'Other Diploma',
                   'Polytechnic',              'No Qualification',
                       'Primary',               'Lower Secondary']
Length: 8, dtype: str


,age,gender,marital_status,education_level,occupation,industry,planning_area,persona,cultural_background,career_goals_and_ambitions,source_index,age_bucket,education_level_processed
2,29,Female,Single,University,Professional,Public Administration & Education Services,Clementi,Betty is a methodical policy pro who balances ...,Wei Qi Leong grew up in a middle‑class Chinese...,Betty aims to advance to a senior policy advis...,2,26-30,Degree and above
5,39,Male,Married,University,Senior Official or Manager,Manufacturing,Sengkang,"Qijie Huat Chua, a night-owl puzzle enthusiast...",Qijie Huat Chua grew up in a Singaporean Chine...,He aims to advance to a C‑suite role such as C...,5,36-40,Degree and above
6,42,Female,Married,Post Secondary (Non-Tertiary),Homemaker,NaN,Jurong West,Wai Ting Soon is a habit‑driven planner who ca...,Wai Ting grew up in a traditional Chinese Sing...,While she values her role as a full‑time homem...,6,41-45,Diploma / A-Level
7,42,Male,Single,University,Professional,Financial & Insurance Services,Queenstown,Chongyong Ang balances a spreadsheet‑obsessed ...,Chongyong grew up in a typical Chinese Singapo...,Chongyong aims to progress to a senior managem...,7,41-45,Degree and above
9,31,Male,Single,University,Senior Official or Manager,Wholesale & Retail Trade,Woodlands,Yao Dar Teo (Daniel) is a data‑savvy retail st...,"Daniel grew up in Woodlands, a residential tow...",Daniel aims to advance to a senior director or...,9,31-35,Degree and above


Sampling using M&P targets

In [8]:
TARGETS = {
    "Single": {
        "n": 49,
        "age_bucket": {
            "21-25": 0.30,
            "26-30": 0.34,
            "31-35": 0.18,
            "36-40": 0.10,
            "41-45": 0.09,
        },
        "gender": {
            "Male": 0.52,
            "Female": 0.48,
        },
        "education_level": {
            "Secondary and below": 0.12,
            "Diploma / A-Level": 0.34,
            "Degree and above": 0.55,
        },
    },
    "Married": {
        "n": 51,
        "age_bucket": {
            "21-25": 0.01,
            "26-30": 0.09,
            "31-35": 0.24,
            "36-40": 0.30,
            "41-45": 0.37,
        },
        "gender": {
            "Male": 0.45,
            "Female": 0.55,
        },
        "education_level": {
            "Secondary and below": 0.16,
            "Diploma / A-Level": 0.28,
            "Degree and above": 0.56,
        },
    }
}

In [9]:
# Step 5: Filter usable rows only

df = df[
    df["marital_status"].isin(["Single", "Married"])
    & df["education_level_processed"].notna()
    & df["age_bucket"].notna()
    & df["gender"].isin(["Male", "Female"])
].copy()

print("Available rows by marital_status:")
print(df["marital_status"].value_counts())

print("\nEducation distribution:")
print(df["education_level_processed"].value_counts())

print("\nGender distribution:")
print(df["gender"].value_counts())

print("\nAge bucket distribution:")
print(df["age_bucket"].value_counts())

Available rows by marital_status:
marital_status
Married    33762
Single     28907
Name: count, dtype: int64

Education distribution:
education_level_processed
Degree and above       33203
Diploma / A-Level      22531
Secondary and below     6935
Name: count, dtype: int64

Gender distribution:
gender
Female    31536
Male      31133
Name: count, dtype: int64

Age bucket distribution:
age_bucket
31-35    13246
26-30    12860
36-40    12707
41-45    12556
21-25    11300
Name: count, dtype: int64


In [10]:
# Step 6: Weighted sample according to M&P targets

RANDOM_STATE = 42

def add_sampling_weights(group_df, group_name):
    group_df = group_df.copy()
    spec = TARGETS[group_name]

    group_df["sampling_weight"] = 1.0

    column_map = {
        "age_bucket": "age_bucket",
        "gender": "gender",
        "education_level": "education_level_processed"
    }

    for target_col, df_col in column_map.items():
        observed_dist = group_df[df_col].value_counts(normalize=True).to_dict()
        target_dist = spec[target_col]

        def get_weight(value):
            observed = observed_dist.get(value, 0)
            target = target_dist.get(value, 0)

            if observed == 0:
                return 0
            
            return target / observed

        group_df["sampling_weight"] *= group_df[df_col].apply(get_weight)

    return group_df


sampled_parts = []

for group_name in ["Single", "Married"]:
    group_df = df[df["marital_status"] == group_name].copy()
    group_df = add_sampling_weights(group_df, group_name)

    n_required = TARGETS[group_name]["n"]

    sampled = group_df.sample(
        n=n_required,
        weights="sampling_weight",
        replace=False,
        random_state=RANDOM_STATE
    )

    sampled_parts.append(sampled)

agents = pd.concat(sampled_parts).sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

agents["agent_id"] = [f"A{i+1:03d}" for i in range(len(agents))]

print("Final sampled agents:", len(agents))
print(agents["marital_status"].value_counts())



Final sampled agents: 100
marital_status
Married    51
Single     49
Name: count, dtype: int64


Step 7: RNG split test

In [12]:
# Step 7: Split Single agents into Single and Dating

rng = np.random.default_rng(RANDOM_STATE)

agents["relationship_status"] = pd.NA
agents["relationship_status_source"] = pd.NA

# Married agents remain Married
married_mask = agents["marital_status"] == "Married"
agents.loc[married_mask, "relationship_status"] = "Married"
agents.loc[married_mask, "relationship_status_source"] = "raw_marital_status"

# Single agents are split into Single and Dating using the survey 50/50 dating split
single_idx = agents[agents["marital_status"] == "Single"].index.to_list()

n_single_total = len(single_idx)
n_dating = n_single_total // 2

dating_idx = rng.choice(
    single_idx,
    size=n_dating,
    replace=False
)

agents.loc[single_idx, "relationship_status"] = "Single"
agents.loc[single_idx, "relationship_status_source"] = "survey_imputed"

agents.loc[dating_idx, "relationship_status"] = "Dating"
agents.loc[dating_idx, "relationship_status_source"] = "survey_imputed"

agents["relationship_status"].value_counts()

relationship_status
Married    51
Single     25
Dating     24
Name: count, dtype: int64

In [13]:
# Step 8: Create final agent table

final_agents = agents[[
    "agent_id",
    "age",
    "age_bucket",
    "gender",
    "marital_status",
    "relationship_status",
    "relationship_status_source",
    "education_level_processed",
    "occupation",
    "industry",
    "planning_area",
    "persona",
    "cultural_background",
    "career_goals_and_ambitions",
    "source_index"
]].copy()

final_agents = final_agents.rename(columns={
    "education_level_processed": "education_level"
})

final_agents.head(10)

,agent_id,age,age_bucket,gender,marital_status,relationship_status,relationship_status_source,education_level,occupation,industry,planning_area,persona,cultural_background,career_goals_and_ambitions,source_index
0,A001,37,36-40,Male,Married,Married,raw_marital_status,Degree and above,Professional,Administrative & Support Services,Jurong West,Clement Hao Oon (Roland) is a curiosity‑driven...,Clement grew up in Jurong West as part of Sing...,He aims to progress to a senior management rol...,142960
1,A002,35,31-35,Female,Married,Married,raw_marital_status,Diploma / A-Level,Associate Professional or Technician,Public Administration & Education Services,Bukit Panjang,"Susan, a 35‑year‑old public‑service profession...",Susan grew up in a typical Chinese Singaporean...,Susan aims to progress to a senior officer rol...,22950
2,A003,40,36-40,Male,Married,Married,raw_marital_status,Degree and above,Senior Official or Manager,Professional Services,Clementi,"Effande Kamsan, 40, is a disciplined yet occas...",Effande is a second‑generation Malay Singapore...,Effande aims to progress to a Director‑level r...,20511
3,A004,28,26-30,Female,Single,Single,survey_imputed,Degree and above,Associate Professional or Technician,Public Administration & Education Services,Yishun,"Vivienne Hoon Goh, 28, couples a meticulous po...",Vivienne grew up in Yishun in a typical Singap...,Vivienne aims to advance to a senior officer p...,98540
4,A005,22,21-25,Female,Single,Single,survey_imputed,Degree and above,Senior Official or Manager,Wholesale & Retail Trade,Kallang,"Annaliza, a 22‑year‑old bullet‑journal‑wieldin...",Annaliza grew up in the vibrant Malay enclave ...,"In the short term, Annaliza aims to master the...",38876
5,A006,21,21-25,Female,Single,Dating,survey_imputed,Diploma / A-Level,Senior Official or Manager,Financial & Insurance Services,Choa Chu Kang,"Man Leng Tong, known as Serene, is a 21‑year‑o...",Serene grew up in a typical Hokkien‑Teochew Ch...,Serene aims to advance to a Chief Risk Officer...,65835
6,A007,27,26-30,Female,Single,Dating,survey_imputed,Diploma / A-Level,Clerical Worker,Public Administration & Education Services,Bukit Panjang,"Li Tan, 27, balances a talkative, detail‑drive...",Li Tan was raised in a Chinese Singaporean hou...,Li Tan aims to progress from her current cleri...,43822
7,A008,42,41-45,Male,Married,Married,raw_marital_status,Degree and above,Professional,Manufacturing,Toa Payoh,Joy is a methodical yet easy‑going manufacturi...,Joy grew up in the mature HDB estate of Toa Pa...,Joy aims to progress to Plant Manager and even...,25332
8,A009,33,31-35,Female,Single,Single,survey_imputed,Degree and above,Professional,Manufacturing,Orchard,Caroline Ho (Vasanti) blends meticulous lean-m...,Caroline (Vasanti) is a Singapore‑born Indian ...,She aims to progress to a senior management ro...,3138
9,A010,40,36-40,Female,Single,Dating,survey_imputed,Diploma / A-Level,Unemployed,NaN,Serangoon,"Renuka, a 40‑year‑old detail‑oriented planner,...",Renuka belongs to the Indian diaspora in Singa...,"Renuka aims to secure a stable, structured rol...",56089


In [14]:
# Step 9: Verify final distributions

print("Relationship status counts:")
display(final_agents["relationship_status"].value_counts())

print("Original marital status counts:")
display(final_agents["marital_status"].value_counts())

print("Age distribution by marital status:")
display(
    pd.crosstab(
        final_agents["marital_status"],
        final_agents["age_bucket"],
        normalize="index"
    ).round(2)
)

print("Gender distribution by marital status:")
display(
    pd.crosstab(
        final_agents["marital_status"],
        final_agents["gender"],
        normalize="index"
    ).round(2)
)

print("Education distribution by marital status:")
display(
    pd.crosstab(
        final_agents["marital_status"],
        final_agents["education_level"],
        normalize="index"
    ).round(2)
)

Relationship status counts:


relationship_status
Married    51
Single     25
Dating     24
Name: count, dtype: int64

Original marital status counts:


marital_status
Married    51
Single     49
Name: count, dtype: int64

Age distribution by marital status:


age_bucket,21-25,26-30,31-35,36-40,41-45
marital_status,,,,,
Married,0.00,0.06,0.29,0.25,0.39
Single,0.33,0.31,0.14,0.10,0.12


Gender distribution by marital status:


gender,Female,Male
marital_status,,
Married,0.55,0.45
Single,0.53,0.47


Education distribution by marital status:


education_level,Degree and above,Diploma / A-Level,Secondary and below
marital_status,,,
Married,0.75,0.14,0.12
Single,0.47,0.37,0.16


In [15]:
# Step 10: Export

final_agents.to_csv("week4_agents_100_marriage_survey_calibrated.csv", index=False)

print("Saved: week4_agents_100_marriage_survey_calibrated.csv")

Saved: week4_agents_100_marriage_survey_calibrated.csv
